# Drift Correction in 1D: A Step-by-Step Tutorial

**Based on:** Hamidian et al., *New J. Phys.* **14**, 053017 (2012)

This notebook walks through the drift correction algorithm used in scanning tunneling microscopy (SI-STM), adapted to **1D signals** for pedagogical clarity. Each step of the math corresponds to one figure.

In STM, long acquisition times (days to weeks) mean that thermal drift and piezo relaxation distort the measured images. The Lawler-Fujita drift correction algorithm extracts the distortion field from the known periodicity of the crystal lattice, then undoes it. Here we demonstrate every step of this process on a simple 1D cosine "lattice."

**Dependencies:** numpy, scipy, matplotlib (no stmpy required)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

plt.rcParams['figure.figsize'] = [6, 4]

## Step 1: The Perfect Lattice

In STM, the topography of a crystal surface is periodic with the lattice. In 1D, the simplest model is a cosine corrugation:

$$A(r') = \cos(k \cdot r')$$

where $k = 2\pi / d$ is the wavevector and $d$ is the lattice constant. The coordinate $r'$ denotes the *true* position in a perfect, undistorted lattice.

In [ ]:
# Parameters
L = 15.0           # scan length (nm)
N = 1000           # number of sample points
d = 2.0            # lattice constant (nm)
k = 2 * np.pi / d  # wavevector (rad/nm)

x = np.linspace(0, L, N)
A_perfect = np.cos(k * x)

atoms = np.arange(0, L, d)  # atom positions at every lattice constant

plt.figure(figsize=[6, 4])
plt.plot(x, A_perfect)
plt.plot(atoms, atoms * 0, 'b.', ms=10, mew=2)
plt.xlim(-1, 11)
plt.ylim(-2, 2)
plt.xlabel("Distance (nm)")
plt.ylabel("Height ($\\AA$)")
plt.title("Step 1: Perfect lattice  $A(r') = \\cos(k \\cdot r')$")
plt.show()

## Step 2: Introduce the Drift Field $u(r)$

In a real STM measurement, the tip drifts over time due to:
- **Thermal drift** — temperature changes cause the scanner to expand/contract
- **Piezo relaxation** — the piezoelectric scanner creeps after a voltage step (logarithmic decay)

This creates a position-dependent displacement field $u(r)$: when the scanner thinks it is at position $r$, the tip is actually at position $r + u(r)$.

We model this with a logarithmic drift (realistic for piezo relaxation):

$$u(r) = \frac{\ln(r + 1)}{k}$$

In [ ]:
# Drift field
theta = lambda r: np.log(r + 1)   # drift phase: theta(r) = k * u(r)
u = lambda r: theta(r) / k        # drift field in nm

plt.figure(figsize=[6, 4])
plt.plot(x, u(x))
plt.xlabel("Distance (nm)")
plt.ylabel("Drift $u(r)$ (nm)")
plt.title("Step 2: Drift field  $u(r) = \\ln(r+1) \\,/\\, k$")
plt.xlim(-1, L + 1)
plt.show()

## Step 3: The Distorted Measurement

Due to the drift field, the signal measured at nominal position $r$ is actually the topography at the true position $r + u(r)$:

$$A(r) = \cos[k \cdot (r + u(r))]$$

The atoms are no longer evenly spaced — they appear compressed where the drift is large (early in the scan) and more correctly spaced where drift is small (later).

In [ ]:
A_distorted = np.cos(k * x + theta(x))  # = cos[k * (r + u(r))]

# Find where atoms appear in the distorted signal
# Atoms are at positions where k*(r + u(r)) = 2*pi*n, i.e. the maxima
# For visualization, mark the perfect vs drifted atom positions
atoms_perfect = np.arange(0, 11, d)

plt.figure(figsize=[6, 4])
plt.plot(x, A_perfect, 'C0', alpha=0.4, label='Perfect lattice')
plt.plot(atoms_perfect, np.cos(k * atoms_perfect), 'b.', ms=12, mew=2)
plt.plot(x, A_distorted, 'r', label='Distorted (measured)')
plt.plot(atoms_perfect, np.cos(k * atoms_perfect + theta(atoms_perfect)), 'r.', ms=8, mew=2)

# Arrows showing how drift shifts each atom
for a in atoms_perfect:
    y_atom = np.cos(k * a + theta(a))
    plt.annotate('', xy=[a, -1.5], xytext=[a, -1.8],
                 arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))

plt.legend(loc='upper right')
plt.xlim(-1, 11)
plt.ylim(-2, 2)
plt.xlabel("Distance (nm)")
plt.ylabel("Height ($\\AA$)")
plt.title("Step 3: $A(r) = \\cos[k(r + u(r))]$")
plt.show()

## Step 4: Rewrite Using the Drifting Phase

We can separate the "ideal lattice" part from the "drift error" by expanding the argument:

$$
\begin{align}
A(r) &= \cos[k \cdot r + k \cdot u(r)] \\
     &= \cos[k \cdot r + \theta(r)]
\end{align}
$$

where $\theta(r) = k \cdot u(r)$ is the **drifting phase**. In practice, we work with $\theta(r)$ rather than $u(r)$ directly, because the phase is what we can extract from the signal.

If we can recover $\theta(r)$, we can undo the drift: transform from the measured coordinate $r$ back to the true coordinate $\tilde{r} = r - u(r)$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=[10, 4])

axes[0].plot(x, u(x))
axes[0].set_xlabel("Distance (nm)")
axes[0].set_ylabel("Drift $u(r)$ (nm)")
axes[0].set_title("Drift field $u(r)$")
axes[0].set_xlim(-1, L + 1)

axes[1].plot(x, theta(x), 'C1')
axes[1].set_xlabel("Distance (nm)")
axes[1].set_ylabel("$\\theta(r)$ (rad)")
axes[1].set_title("Drift phase $\\theta(r) = k \\cdot u(r)$")
axes[1].set_xlim(-1, L + 1)

plt.tight_layout()
plt.show()

## Step 5: FFT of the Distorted Signal

The Fourier transform reveals the spatial frequency content of our signal:
- A **perfect** cosine has a sharp peak at frequency $k$ (in 2D STM, this would be called a "Bragg peak" — here it is simply the Fourier peak at the lattice frequency $1/d$).
- **Drift broadens** this peak because the effective local spacing varies across the signal: compressed in some regions, stretched in others.

This broadening is the signature of drift distortion, and motivates the need for correction.

In [ ]:
# Compute FFTs
freqs = np.fft.fftshift(np.fft.fftfreq(N, d=x[1] - x[0]))  # spatial frequency (1/nm)
fft_perfect = np.abs(np.fft.fftshift(np.fft.fft(A_perfect)))
fft_distorted = np.abs(np.fft.fftshift(np.fft.fft(A_distorted)))

fig, axes = plt.subplots(1, 2, figsize=[10, 4])

axes[0].plot(freqs, fft_perfect, 'C0')
axes[0].set_xlabel("Spatial frequency (1/nm)")
axes[0].set_ylabel("|FFT|")
axes[0].set_title("FFT of perfect lattice")
axes[0].set_xlim(-2, 2)

axes[1].plot(freqs, fft_distorted, 'r')
axes[1].set_xlabel("Spatial frequency (1/nm)")
axes[1].set_ylabel("|FFT|")
axes[1].set_title("FFT of distorted lattice")
axes[1].set_xlim(-2, 2)

plt.tight_layout()
plt.show()

## Step 6: Computational Lock-in Detection

**Goal:** Extract the unknown drift phase $\theta(r)$ from the measured signal $A(r) = \cos[kr + \theta(r)]$.

The trick is to multiply by reference cosine and sine at the *known* lattice wavevector $k$. Using the product-to-sum trig identity:

$$A(r) \cdot \cos(kr) = \cos[kr + \theta(r)] \cdot \cos(kr) = \frac{1}{2}\cos(\theta(r)) + \frac{1}{2}\cos(2kr + \theta(r))$$

$$A(r) \cdot \sin(kr) = \cos[kr + \theta(r)] \cdot \sin(kr) = \frac{1}{2}\sin(\theta(r)) + \frac{1}{2}\sin(2kr + \theta(r))$$

Each product has two parts:
- A **slowly varying** part that depends only on $\theta(r)$ — this is what we want
- A **fast oscillation** at $2k$ — this we will filter out

This is the 1D version of Hamidian Eqs. (1a)/(1b), which in 2D use two Bragg wavevectors $\mathbf{Q}_a$ and $\mathbf{Q}_b$.

In [ ]:
# Lock-in: multiply distorted signal by reference cos and sin
ref_cos = np.cos(k * x)
ref_sin = np.sin(k * x)

prod_cos = A_distorted * ref_cos  # (1/2)cos(theta) + (1/2)cos(2kr + theta)
prod_sin = A_distorted * ref_sin  # (1/2)sin(theta) + (1/2)sin(2kr + theta)

fig, axes = plt.subplots(2, 1, figsize=[8, 6], sharex=True)

axes[0].plot(x, prod_cos, 'C0', alpha=0.7, label='$A(r) \\cdot \\cos(kr)$')
axes[0].plot(x, 0.5 * np.cos(theta(x)), 'k--', lw=2,
             label='$\\frac{1}{2}\\cos(\\theta(r))$  (slow part)')
axes[0].set_ylabel("Amplitude")
axes[0].legend()
axes[0].set_title("Step 6: Lock-in products")

axes[1].plot(x, prod_sin, 'C1', alpha=0.7, label='$A(r) \\cdot \\sin(kr)$')
axes[1].plot(x, 0.5 * np.sin(theta(x)), 'k--', lw=2,
             label='$\\frac{1}{2}\\sin(\\theta(r))$  (slow part)')
axes[1].set_xlabel("Distance (nm)")
axes[1].set_ylabel("Amplitude")
axes[1].legend()

plt.xlim(-1, L + 1)
plt.tight_layout()
plt.show()

## Step 7: Low-Pass Filter

To isolate the slowly varying part (containing $\theta(r)$) from the fast $2k$ oscillation, we apply a **low-pass filter** — here, a Gaussian convolution.

The filter width $\sigma$ must satisfy:
- $\sigma \gg d$ (lattice spacing) — to average out the $2k$ oscillation
- $\sigma \ll$ scale of drift variation — to preserve the spatial dependence of $\theta(r)$

$$X(r) = \text{LPF}[A \cdot \cos(kr)] \approx \tfrac{1}{2}\cos(\theta(r))$$
$$Y(r) = \text{LPF}[A \cdot \sin(kr)] \approx \tfrac{1}{2}\sin(\theta(r))$$

This is the 1D equivalent of the Gaussian mask convolution used in the 2D `stmpy.driftcorr` algorithm.

In [ ]:
# Gaussian low-pass filter
# One lattice period d = 2 nm spans d / (L/N) = 2 / 0.015 ≈ 133 pixels
# sigma should be comparable to or larger than one lattice period
sigma_pixels = 80  # ≈ 1.2 nm

X_r = gaussian_filter1d(prod_cos, sigma=sigma_pixels)  # ≈ (1/2)cos(theta)
Y_r = gaussian_filter1d(prod_sin, sigma=sigma_pixels)  # ≈ (1/2)sin(theta)

fig, axes = plt.subplots(2, 1, figsize=[8, 6], sharex=True)

axes[0].plot(x, prod_cos, 'C0', alpha=0.3, label='Raw product')
axes[0].plot(x, X_r, 'C0', lw=2, label='$X(r)$ = LPF[$A \\cdot \\cos(kr)$]')
axes[0].plot(x, 0.5 * np.cos(theta(x)), 'k--', lw=1,
             label='True $\\frac{1}{2}\\cos(\\theta)$')
axes[0].legend()
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Step 7: Low-pass filtered lock-in outputs")

axes[1].plot(x, prod_sin, 'C1', alpha=0.3, label='Raw product')
axes[1].plot(x, Y_r, 'C1', lw=2, label='$Y(r)$ = LPF[$A \\cdot \\sin(kr)$]')
axes[1].plot(x, 0.5 * np.sin(theta(x)), 'k--', lw=1,
             label='True $\\frac{1}{2}\\sin(\\theta)$')
axes[1].legend()
axes[1].set_xlabel("Distance (nm)")
axes[1].set_ylabel("Amplitude")

plt.xlim(-1, L + 1)
plt.tight_layout()
plt.show()

## Step 8: Extract the Phase $\theta(r)$

From the filtered lock-in outputs $X(r) \approx \tfrac{1}{2}\cos\theta$ and $Y(r) \approx \tfrac{1}{2}\sin\theta$, we recover the phase:

$$\theta(r) = \arctan\!\left(\frac{Y(r)}{X(r)}\right)$$

This is the 1D version of Hamidian Eqs. (2a)/(2b):

$$\theta_a(\mathbf{r}) = \tan^{-1}\frac{Y_a(\mathbf{r})}{X_a(\mathbf{r})}$$

We use `np.arctan2` for the full $(-\pi, \pi]$ range, followed by `np.unwrap` to handle any $2\pi$ discontinuities.

In [ ]:
# Extract phase
theta_extracted = np.arctan2(Y_r, X_r)
theta_extracted = np.unwrap(theta_extracted)  # handle any 2*pi jumps

plt.figure(figsize=[6, 4])
plt.plot(x, theta(x), 'k', lw=2, label='True $\\theta(r) = \\ln(r+1)$')
plt.plot(x, theta_extracted, 'r--', lw=2, label='Extracted $\\theta(r)$')
plt.xlabel("Distance (nm)")
plt.ylabel("Phase (rad)")
plt.title("Step 8: Phase extraction — Eqs. (2a)/(2b) in 1D")
plt.legend()
plt.xlim(-1, L + 1)
plt.show()

## Step 9: Recover the Drift Field $u(r)$

Since $\theta(r) = k \cdot u(r)$, we simply divide:

$$u(r) = \frac{\theta(r)}{k}$$

In 2D (Hamidian Eq. 9), this becomes a matrix equation because there are two Bragg wavevectors and two spatial components of drift:

$$\mathbf{u}(\mathbf{r}) = \mathbf{Q}^{-1} \begin{pmatrix} \bar{\theta}_a - \theta_a(\mathbf{r}) \\ \bar{\theta}_b - \theta_b(\mathbf{r}) \end{pmatrix}$$

In 1D, $\mathbf{Q}^{-1}$ reduces to $1/k$.

In [ ]:
# Recover drift field
u_extracted = theta_extracted / k

plt.figure(figsize=[6, 4])
plt.plot(x, u(x), 'k', lw=2, label='True $u(r)$')
plt.plot(x, u_extracted, 'r--', lw=2, label='Recovered $u(r)$')
plt.xlabel("Distance (nm)")
plt.ylabel("Drift (nm)")
plt.title("Step 9: Recovered drift field — Eq. (9) in 1D")
plt.legend()
plt.xlim(-1, L + 1)
plt.show()

## Step 10: Apply the Correction — Undistort the Signal

Now we undo the drift. The corrected coordinate is:

$$\tilde{r} = r - u(r)$$

We know the distorted signal $A(r)$ at each nominal position $r$. We compute the corrected position $\tilde{r} = r - u(r)$ for each $r$, giving us pairs $(\tilde{r}, A)$. Then we interpolate these onto a regular grid to get the corrected signal $A'(\tilde{r})$.

In the 2D algorithm, this same operation is done with 2D interpolation: each pixel at $(x, y)$ is remapped to $(x - u_x, y - u_y)$.

In [ ]:
# Corrected positions
r_tilde = x - u_extracted  # corrected coordinates

# Interpolate: we have (r_tilde, A_distorted) pairs, resample onto regular grid x
A_corrected = np.interp(x, r_tilde, A_distorted)

fig, axes = plt.subplots(3, 1, figsize=[8, 8], sharex=True)

axes[0].plot(x, A_perfect, 'C0')
axes[0].set_ylabel("Height ($\\AA$)")
axes[0].set_title("Perfect lattice")

axes[1].plot(x, A_distorted, 'r')
axes[1].set_ylabel("Height ($\\AA$)")
axes[1].set_title("Distorted lattice (measured)")

axes[2].plot(x, A_corrected, 'C2', label='Corrected')
axes[2].plot(x, A_perfect, 'C0', alpha=0.4, ls='--', label='Perfect (reference)')
axes[2].set_ylabel("Height ($\\AA$)")
axes[2].set_xlabel("Distance (nm)")
axes[2].set_title("Corrected lattice")
axes[2].legend()

for ax in axes:
    ax.set_xlim(-1, L + 1)
    ax.set_ylim(-2, 2)

plt.tight_layout()
plt.show()

In [ ]:
# FFT comparison: before and after correction
fft_corrected = np.abs(np.fft.fftshift(np.fft.fft(A_corrected)))

fig, axes = plt.subplots(1, 3, figsize=[12, 4])

for ax, fft_data, title, color in zip(
    axes,
    [fft_perfect, fft_distorted, fft_corrected],
    ['Perfect', 'Distorted', 'Corrected'],
    ['C0', 'r', 'C2']
):
    ax.plot(freqs, fft_data, color)
    ax.set_title(f"FFT: {title}")
    ax.set_xlim(-2, 2)
    ax.set_xlabel("Spatial frequency (1/nm)")
    ax.set_ylabel("|FFT|")

plt.tight_layout()
plt.show()

## Step 11: Transfer the Correction to a Second Signal

This is the key practical insight from Hamidian (Section 1, final paragraph):

> *"The identical geometrical transformations to undo $u(\mathbf{r})$ in $T(\mathbf{r})$ yielding $T'(\mathbf{r})$ are also carried out on every $g(\mathbf{r}, V)$ acquired simultaneously."*

In STM practice:
1. Derive $u(\mathbf{r})$ from the **topography** $T(\mathbf{r})$ (which has known lattice periodicity)
2. Apply the **same** correction to all **dI/dV maps** $g(\mathbf{r}, V)$ (which were measured simultaneously)

Here we demonstrate this with a second signal $B(r)$ that has a different spatial frequency but is distorted by the **same** drift field $u(r)$.

In [ ]:
# Second signal: different frequency, same drift
d2 = 3.0                   # different lattice constant (nm)
k2 = 2 * np.pi / d2

B_perfect = np.sin(k2 * x)
B_distorted = np.sin(k2 * (x + u(x)))  # same drift field u(r)!

# Apply the SAME correction (derived from signal A) to signal B
B_corrected = np.interp(x, r_tilde, B_distorted)

fig, axes = plt.subplots(3, 1, figsize=[8, 8], sharex=True)

axes[0].plot(x, B_perfect, 'C0')
axes[0].set_title("Signal B: Perfect (different frequency, $d$ = 3 nm)")

axes[1].plot(x, B_distorted, 'r')
axes[1].set_title("Signal B: Distorted by same $u(r)$")

axes[2].plot(x, B_corrected, 'C2', label='Corrected')
axes[2].plot(x, B_perfect, 'C0', alpha=0.4, ls='--', label='Perfect (reference)')
axes[2].set_title("Signal B: Corrected using drift from Signal A")
axes[2].legend()

for ax in axes:
    ax.set_ylabel("Amplitude")
    ax.set_xlim(-1, L + 1)
    ax.set_ylim(-2, 2)

axes[2].set_xlabel("Distance (nm)")
plt.tight_layout()
plt.show()

In [ ]:
# FFT of second signal: before and after
fft_B_perfect = np.abs(np.fft.fftshift(np.fft.fft(B_perfect)))
fft_B_distorted = np.abs(np.fft.fftshift(np.fft.fft(B_distorted)))
fft_B_corrected = np.abs(np.fft.fftshift(np.fft.fft(B_corrected)))

fig, axes = plt.subplots(1, 3, figsize=[12, 4])

for ax, fft_data, title, color in zip(
    axes,
    [fft_B_perfect, fft_B_distorted, fft_B_corrected],
    ['Perfect B', 'Distorted B', 'Corrected B'],
    ['C0', 'r', 'C2']
):
    ax.plot(freqs, fft_data, color)
    ax.set_title(f"FFT: {title}")
    ax.set_xlim(-2, 2)
    ax.set_xlabel("Spatial frequency (1/nm)")
    ax.set_ylabel("|FFT|")

plt.tight_layout()
plt.show()

## Summary: From 1D to 2D

We demonstrated the complete Lawler-Fujita drift correction algorithm in 1D:

| Step | 1D | 2D (Hamidian) |
|------|-----|---------------|
| 1. Perfect signal | $A(r') = \cos(kr')$ | $T(\mathbf{r}') = \cos(\mathbf{Q}_a \cdot \mathbf{r}') + \cos(\mathbf{Q}_b \cdot \mathbf{r}')$ |
| 2. Drift field | Scalar $u(r)$ | Vector $\mathbf{u}(\mathbf{r}) = (u_x, u_y)$ |
| 3. Distorted signal | $\cos[k(r + u(r))]$ | $\cos[\mathbf{Q}_a \cdot (\mathbf{r} + \mathbf{u})]$ + ... |
| 4. Drift phase | $\theta = k \cdot u$ | $\theta_a = \mathbf{Q}_a \cdot \mathbf{u}$, $\theta_b = \mathbf{Q}_b \cdot \mathbf{u}$ |
| 5. FFT | 1D FFT peak broadened | 2D FFT Bragg peaks broadened |
| 6. Lock-in | Multiply by $\cos(kr)$, $\sin(kr)$ | Multiply by $\cos(\mathbf{Q} \cdot \mathbf{r})$, $\sin(\mathbf{Q} \cdot \mathbf{r})$ — Eqs. (1a/1b) |
| 7. Low-pass filter | 1D Gaussian | 2D Gaussian convolution |
| 8. Phase extraction | $\theta = \arctan(Y/X)$ | $\theta_a, \theta_b$ — Eqs. (2a/2b) |
| 9. Recover drift | $u = \theta / k$ | $\mathbf{u} = \mathbf{Q}^{-1} \boldsymbol{\theta}$ — Eq. (9) |
| 10. Correct | `np.interp` | 2D interpolation (`interp2d`) |
| 11. Transfer | Same $u$ for signal B | Same $\mathbf{u}$ for all $g(\mathbf{r}, V)$ |

The full 2D implementation is available in `stmpy.driftcorr` and demonstrated in `Stmpy_102_dfc_tutorial_dev.ipynb`.

**Reference:** Hamidian et al., "Picometer registration of zinc impurity states in Bi$_2$Sr$_2$CaCu$_2$O$_{8+\delta}$", *New J. Phys.* **14**, 053017 (2012)